# 03 – Model Training
This notebook trains the selected models and persists *all* artefacts. It delegates **all heavy‑lifting** to the unified `src.training.run_training` helper.

In [1]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

from pathlib import Path
import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

Repository Root: /home/marcmaceira/projects/reuters-rag-classifier_clean
Configuration: {'general': {'run_name': 'experiment_with_13_classes', 'seed': 42, 'n_classes': 13}, 'dataset': {'split_type': 'test', 'test_split': 0.2, 'cutoff_year': 1996}, 'paths': {'data_exploration_dir': 'experiment_with_13_classes/data_exploration', 'artifacts_dir': 'experiment_with_13_classes/artifacts', 'embeddings_dir': 'experiment_with_13_classes/embeddings', 'models_dir': 'experiment_with_13_classes/models', 'results_dir': 'experiment_with_13_classes/results'}, 'model': {'embedding_backend': 'sbert', 'sbert_model_name': 'sentence-transformers/all-MiniLM-L12-v2', 'openai_model_name': 'text-embedding-3-small', 'classifier': 'linear_svm', 'rag_top_k': 5, 'use_llm_refine': False}, 'training': {'batch_size': 32, 'max_epochs': 10, 'learning_rate': '1e-3'}, 'evaluation': {'metrics': ['accuracy', 'macro_f1', 'auc_ovr']}}

=== Configuration Variables ===

[DATASET]
  DATASET_CUTOFF_YEAR: 1996
  DATASET_SPLIT_TYP

In [2]:
from config.notebook_setup import *

# Your helpers (get_dataset, model classes…) live in *src*
from src.datasets.dataset import get_dataset
from src.training import run_training

In [3]:


# Load dataset
# X_train, y_train, _, _, _ = get_dataset(split_type="standard", n_classes=N_CLASSES)
X_train, y_train, X_test, y_test, classes = get_dataset(
    split_type=DATASET_SPLIT_TYPE,
    n_classes=N_CLASSES,
    cutoff_year=GENERAL_CUTOFF_YEAR
)

print(f"Loaded {len(X_train)} training documents with {N_CLASSES} classes")


INFO | Loading Reuters dataset with configuration:
INFO |   - Split type: test
INFO |   - Number of classes: 13
INFO |   - Samples per class: 20
INFO |   - Random seed: None
INFO | Loading small test dataset with 20 samples per class across 13 classes
INFO | Selected classes: earn, acq, crude, interest, money-fx, trade, grain, corn, dlr, money-supply, ship, coffee, sugar
INFO |   - Class 'earn': 14 train, 6 test
INFO |   - Class 'acq': 14 train, 6 test
INFO |   - Class 'crude': 14 train, 6 test
INFO |   - Class 'interest': 14 train, 6 test
INFO |   - Class 'money-fx': 14 train, 6 test
INFO |   - Class 'trade': 14 train, 6 test
INFO |   - Class 'grain': 14 train, 6 test
INFO |   - Class 'corn': 14 train, 6 test
INFO |   - Class 'dlr': 14 train, 6 test
INFO |   - Class 'money-supply': 14 train, 6 test
INFO |   - Class 'ship': 14 train, 6 test
INFO |   - Class 'coffee': 14 train, 6 test
INFO |   - Class 'sugar': 14 train, 6 test


Loaded 182 training documents with 13 classes


In [4]:
import os
from src.datasets.dataset import get_dataset
from src.algorithms.naive_bayes import NaiveBayesClassifier
from src.algorithms.linear_svm import LinearSVMClassifier, LinearSVMBigrams
from src.algorithms.transformer_logreg import TransformerLogReg
from src.rag import load_kmajority, load_centroid, load_llm
from src.rag.adapter_sklearn import RagSklearnAdapter
from src.embeddings.openai_embedder import OpenAIEmbedder
from src.rag.vector_store import VectorStore


# ensure we’re using OpenAI for embeddings
os.environ['USE_OPENAI_EMBEDDINGS'] = '1'
# OR explicitly pass use_openai=True below

# instantiate the OpenAI embedder (batch size adjustable)
openai_embedder = OpenAIEmbedder(model="text-embedding-3-small", batch_size=50)


/home/marcmaceira/projects/reuters-rag-classifier_clean/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO | Loading faiss with AVX2 support.
INFO | Successfully loaded faiss with AVX2 support.
INFO | Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes. This is only an error if you're trying to use GPU Faiss.


In [5]:
# -------------------------------------------------------------
# 2. Define models
# -------------------------------------------------------------
models = {
    'Naive Bayes':       NaiveBayesClassifier(),
    'Linear SVM':        LinearSVMClassifier(),
    'TF-IDF bigrams + SVM': LinearSVMBigrams(),
    'MiniLM + LogReg':   TransformerLogReg(),
    # RAG variants all take an embedder under the hood:
    'RAG-kMajority':     RagSklearnAdapter(load_kmajority(top_k=5, use_openai=False)),
    'RAG-CentroidNN':    RagSklearnAdapter(load_centroid()),
    # For the LLM‐backed RAG we also pass the same embedder plus your LLM choice:
    'RAG-LLM (OpenAI-embeddings)': RagSklearnAdapter(
        load_llm(
            top_k=5,
            model="gpt-4o-mini",
            embedder=openai_embedder,
            use_openai=True  # Add this parameter to use the OpenAI index
        )
    ),
    # Add the local embeddings variant:
    'RAG-LLM (local-embeddings)': RagSklearnAdapter(
        load_llm(
            top_k=5,
            model="gpt-4o-mini",
            use_openai=False,  # Explicitly specify to use local index
            embedder=lambda texts: VectorStore.embed("sentence-transformers/all-MiniLM-L6-v2", texts)
        )
    ),
}

INFO | Use pytorch device_name: cpu
INFO | Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO | Loading SentenceTransformer retriever
INFO | Using index: /home/marcmaceira/projects/reuters-rag-classifier_clean/experiment_with_13_classes/embeddings/sbert/index.faiss, meta: /home/marcmaceira/projects/reuters-rag-classifier_clean/experiment_with_13_classes/embeddings/sbert/meta.jsonl
INFO | Loading FAISS index from /home/marcmaceira/projects/reuters-rag-classifier_clean/experiment_with_13_classes/embeddings/sbert/index.faiss
INFO | Loading metadata from /home/marcmaceira/projects/reuters-rag-classifier_clean/experiment_with_13_classes/embeddings/sbert/meta.jsonl
INFO | VectorStore initialized with 182 documents in 0.05 seconds
INFO | Default retriever loaded in 0.06 seconds
INFO | Loading SentenceTransformer retriever
INFO | Using index: /home/marcmaceira/projects/reuters-rag-classifier_clean/experiment_with_13_classes/embeddings/sbert/index.faiss, meta: /home/marcmaceira/projects

In [6]:

# -------------------------------------------------------------
# 3. Train + persist
# -------------------------------------------------------------
trained = run_training(
    models,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    output_dir=MODELS_DIR,
)


[run_training] Fitting Naive Bayes…
[run_training] Fitting Linear SVM…
[run_training] Fitting TF-IDF bigrams + SVM…
[run_training] Fitting MiniLM + LogReg…


INFO | Use pytorch device_name: cpu
INFO | Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO | Encoding 182 documents for training
Batches: 100%|██████████| 3/3 [00:01<00:00,  1.68it/s]

[run_training] Fitting RAG-kMajority…



INFO | Generating embeddings for 182 documents
INFO | Embeddings will be stored in: /home/marcmaceira/projects/reuters-rag-classifier_clean/experiment_with_13_classes/embeddings
INFO | Using SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO | Use pytorch device_name: cpu
INFO | Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO | Use pytorch device_name: cpu
INFO | Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO | Loaded new SentenceTransformer model in 1.39 seconds
INFO | Generated 182 embeddings (shape: (182, 384)) in 6.29 seconds
INFO | Encoding rate: 37.2 docs/second
INFO | Generating embeddings for 182 documents
INFO | Embeddings will be stored in: /home/marcmaceira/projects/reuters-rag-classifier_clean/experiment_with_13_classes/embeddings
INFO | Using SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO | Using cached SentenceTransformer model: sentence-transformers/all-MiniLM-L6-v2
INFO |

[run_training] Fitting RAG-CentroidNN…


INFO | Generating embeddings for 182 documents
INFO | Embeddings will be stored in: /home/marcmaceira/projects/reuters-rag-classifier_clean/experiment_with_13_classes/embeddings
INFO | Using SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO | Using cached SentenceTransformer model: sentence-transformers/all-MiniLM-L6-v2
INFO | Generated 182 embeddings (shape: (182, 384)) in 4.50 seconds
INFO | Encoding rate: 40.4 docs/second
INFO | Generating embeddings for 182 documents
INFO | Embeddings will be stored in: /home/marcmaceira/projects/reuters-rag-classifier_clean/experiment_with_13_classes/embeddings
INFO | Using SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO | Using cached SentenceTransformer model: sentence-transformers/all-MiniLM-L6-v2
INFO | Generated 182 embeddings (shape: (182, 384)) in 4.49 seconds
INFO | Encoding rate: 40.6 docs/second
INFO | Generating embeddings for 78 documents
INFO | Embeddings will be stored in: /home/marcmaceira/projects/re

[run_training] Fitting RAG-LLM (OpenAI-embeddings)…


INFO | Starting prediction for 182 documents with batch size 8
INFO | Starting OpenAI embedding generation for 182 texts with model text-embedding-3-small
INFO | Processing embedding batch 1 with 50 texts
INFO | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO | Batch 1 used 13556 tokens
INFO | Batch 1 completed in 1.64 seconds
INFO | Average time per text in batch: 0.0328 seconds
INFO | Adding delay of 0.29s before next batch
INFO | Processing embedding batch 2 with 50 texts
INFO | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO | Batch 2 used 14842 tokens
INFO | Batch 2 completed in 1.76 seconds
INFO | Average time per text in batch: 0.0351 seconds
INFO | Adding delay of 0.50s before next batch
INFO | Processing embedding batch 3 with 50 texts
INFO | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO | Batch 3 used 10196 tokens
INFO | Batch 3 completed in 1.31 seconds
INFO | Average time per text

[run_training] Fitting RAG-LLM (local-embeddings)…


INFO | Starting prediction for 182 documents with batch size 8
INFO | Generating embeddings for 182 documents
INFO | Embeddings will be stored in: /home/marcmaceira/projects/reuters-rag-classifier_clean/experiment_with_13_classes/embeddings
INFO | Using SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO | Using cached SentenceTransformer model: sentence-transformers/all-MiniLM-L6-v2
INFO | Generated 182 embeddings (shape: (182, 384)) in 4.47 seconds
INFO | Encoding rate: 40.8 docs/second
INFO | Split into 23 batches
INFO | Processing batch 1/23 with 8 documents
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | Processing batch 2/23 with 8 documents
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | Processing batch 3/23 with 8 documents
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | Processing batch 4/23 with 8 documents
INFO | HTTP Request